<a href="https://colab.research.google.com/github/lnittala1-ops/BAN482/blob/main/BAN482_Agentic_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Building our first AI agent
We will build an AI agent that will use a user defined tool to answer a query

### Setup

In [ ]:
# ==============================================================================
# INSTALL REQUIRED LIBRARIES
# ==============================================================================
# - langchain: Core framework for working with Large Language Models (LLMs).
# - langchain-google-genai: Connects LangChain to Google's Gemini models.
# - langgraph: Orchestration library used to build the agent's ReAct or "Reason-Act-Observe" loop.
# The '-q' flag install quietly, without showing progress updates during installation
!pip install -q langchain langchain-google-genai langgraph

# Ignore the error.

In [ ]:
# ==============================================================================
# IMPORT LIBRARIES
# ==============================================================================
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain.agents import create_agent

In [ ]:
# ==============================================================================
# RETRIEVE API KEY SECURELY
# ==============================================================================
# Retrieve your Gemini API key from Colab's "Secrets" menu (key icon on the left).
# Ensure the secret is named 'GEMINI_API_KEY' and access is toggled ON.
api_key = userdata.get('GEMINI_API_KEY')

### Define Tools

In [ ]:
# ==============================================================================
#  DEFINE TOOLS (WHAT THE AGENT CAN DO)
# ==============================================================================
# Standard LLMs only predict text and are notoriously unreliable at exact math.
# By giving the LLM a "tool," we allow it to delegate tasks to reliable Python code.
#

@tool
def multiply(a: int, b: int) -> int:
    """Multiplies two integers together. Use this whenever you need to perform multiplication."""
    return a * b

# CRITICAL RULE FOR AGENTS:
# 1. Type hints (a: int, b: int) tell the LLM what kind of data to pass into the function.
# 2. The docstring ("""...""") acts as the instruction manual. The LLM reads this text
#    to decide *when* and *why* to use this tool.

In [ ]:
# ==============================================================================
# ASSEMBLE TOOLS INTO A LIST
# ==============================================================================
# Put all available tools into a list. An agent can have access to dozens of tools,
# and it will dynamically choose the right one based on the user's question.
tools = [multiply]

### Setup the LLM

In [ ]:
# ==============================================================================
# INITIALIZE THE LLM WITH AN EXPLICIT API KEY
# ==============================================================================
# Instead of relying on environment variables, we pass our `api_key` directly.
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", api_key=api_key)

### Assemble to create the Agent

In [ ]:
# ==============================================================================
# STEP 5: ASSEMBLE THE AGENT (BRAIN + TOOLS + LOOP)
# ==============================================================================
# create_agent builds a "ReAct" (Reason + Act) loop. Behind the scenes, it creates
# a workflow graph that follows these steps:
#   1. REASON: The LLM receives the user's prompt along with tool information from the agent orchestrator and decides if it needs a tool.
#   2. ACT: If yes, the system pauses the LLM and tells the orchestrator to run the Python function (`multiply`).
#   3. OBSERVE: The orchestrator executes the tool as per the LLM's instructions and the Python output is fed back to the LLM.
#   4. RESPOND: The LLM uses that exact result to write a natural language answer.
agent = create_agent(llm, tools)

### Agent is ready for use!

In [ ]:
# ==============================================================================
# RUN THE AGENT & INSPECT THE EXECUTION (WITH VISUAL COMMUNICATION FLOW)
# ==============================================================================
query = "What is 42 multiplied by 7?"
# query = "What is the capital city of USA?"

# .invoke() starts the agent loop.
response = agent.invoke({"messages": [("user", query)]})



In [ ]:
type(response)

In [ ]:
response

In [ ]:
# The final answer to the user is always the last message in the conversation history
print("\n--- Final Answer ---")
final_msg = response["messages"][-1]
if isinstance(final_msg.content, list):
    # Extract text content if it's formatted as content blocks
    for block in final_msg.content:
        if isinstance(block, dict) and block.get("type") == "text":
            print(block["text"])
else:
    print(final_msg.content)

### Looking behind the scenes (Optional)

In [ ]:
print("=== COMMUNICATIONS FLOW ===\n")

for i, message in enumerate(response["messages"]):
    print(f"--- Message #{i+1} ---")

    if message.type == "human":
        print("📬 [USER -> AGENT framework]")
        print(f"User sent query: {message.content}\n")

    elif message.type == "ai":
        if message.tool_calls:
            print("🧠 [LLM -> AGENT framework]")
            print(f"The LLM decided to call a tool: {message.tool_calls}\n")
        else:
            print("📢 [LLM -> USER]")
            print(f"The LLM sent a final text response: {message.content}\n")

    elif message.type == "tool":
        print("⚙️ [AGENT framework -> LLM]")
        print(f"The local Python tool executed and sent the output back to the LLM: {message.content}\n")

print("===========================")
# The final answer to the user is always the last message in the conversation history
print("\n--- Final Answer ---")
final_msg = response["messages"][-1]
if isinstance(final_msg.content, list):
    # Extract text content if it's formatted as content blocks
    for block in final_msg.content:
        if isinstance(block, dict) and block.get("type") == "text":
            print(block["text"])
else:
    print(final_msg.content)
  ###############################